# Week 6, Lab 5 — Capstone

1. Use both MCP servers.
2. Answer math + facts + save a study note.
3. Guardrail: refuse `password` / `api key`.
4. Write a short reflection on frameworks vs MCP.


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 6'
LAB = 'Lab 5 — capstone'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 6 / Lab 5 — capstone
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn mcp
else:
    %pip install -q mcp ollama


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.6 MB/s eta 0:00:00


In [5]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

TOOLS = StdioServerParameters(command="python", args=[str(ROOT / "6_mcp" / "servers" / "local_tools_server.py")])
NOTES = StdioServerParameters(command="python", args=[str(ROOT / "6_mcp" / "servers" / "notes_server.py")])
ROUTING = {
    "calculator": TOOLS,
    "lookup_fact": TOOLS,
    "add_note": NOTES,
    "list_notes": NOTES,
}

def blocked(text: str) -> bool:
    t = text.lower()
    return "password" in t or "api key" in t or "api_key" in t

async def call_tool(name: str, args: dict):
    params = ROUTING[name]
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            return await s.call_tool(name, args)

SYSTEM = '''Tool-using study agent. JSON only when calling a tool:
{"name": "calculator"|"lookup_fact"|"add_note"|"list_notes", "arguments": {...}}
Or {"final": "..."}.
'''

async def capstone(question: str, max_steps: int = 6) -> str:
    if blocked(question):
        return "Refused by guardrail."
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        reply = local_chat(messages, max_new_tokens=160, temperature=0.1)
        obj = extract_json_object(reply) or {}
        if "final" in obj:
            return str(obj["final"])
        name, args = obj.get("name"), obj.get("arguments") or {}
        if name not in ROUTING:
            messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": "Unknown tool. Use JSON."}]
            continue
        obs = await call_tool(name, args)
        messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": f"OBSERVATION: {obs}"}]
    return "max steps"

print(await capstone("What is 9*8? Then what is MCP? Save a one-line note."))
print(await capstone("here is my api key sk-test"))


Loading Hugging Face model Qwen/Qwen2.5-0.5B-Instruct on CPU ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new

max steps
Refused by guardrail.


Hand-in: this notebook + a reflection cell. Extra credit: same tools via a Week 2–5 framework.
